In [1]:
import numpy as np
import pandas as pd
from joblib import load
from utils.paths import DATASETS_PREPROCESSED_DIR, PROJECT_ROOT
from scipy.stats import shapiro

pp_models_dir = PROJECT_ROOT / "power_prediction" / "models"
ar_models_dir = PROJECT_ROOT / "autoregression-models" / "models"

alpha = 0.05

In [2]:
TARGET_LAGS = [1, 2, 3, 4, 5, 6, 24, 48]
FEATURE_LAGS = {
    "temperature_2m": [1, 17],
    "relative_humidity_2m": [6, 30],
    "wind_speed_10m": [6, 30],
    "wind_gusts_10m": [6, 30],
}


def add_lags(df, horizon):
    lagged = df.copy()
    lagged["hour"] = pd.to_datetime(lagged["time"]).dt.hour
    lagged["dayofweek"] = pd.to_datetime(lagged["time"]).dt.dayofweek
    lagged["month"] = pd.to_datetime(lagged["time"]).dt.month
    lagged["row_idx"] = np.arange(len(lagged))

    for lag in TARGET_LAGS:
        lagged[f"delta_p_lag_{lag}"] = lagged["delta_p"].shift(lag)

    for feature, lags in FEATURE_LAGS.items():
        for lag in lags:
            lagged[f"{feature}_lag_{lag}"] = lagged[feature].shift(lag)

    lagged["target"] = lagged["delta_p"].shift(-horizon)
    lagged = lagged.dropna().reset_index(drop=True)

    return lagged

In [ ]:
rows = []

for loc in range(1, 5):
    df = pd.read_csv(DATASETS_PREPROCESSED_DIR / f"Location_{loc}_test.csv")

    X_pp = df.drop(columns=["time", "p_base", "delta_p"])
    y_pp_all = load(pp_models_dir / f"stacking_model{loc}.pkl").predict(X_pp)

    for horizon in [24, 48]:
        lagged = add_lags(df, horizon)
        ar_model = load(ar_models_dir / f"stacking_regressor_h{horizon}.pkl")

        X_ar = lagged.drop(columns=["time", "delta_p", "target", "p_base", "row_idx"])
        y = lagged["target"].to_numpy()
        y_ar = ar_model.predict(X_ar)
        y_pp = np.asarray(y_pp_all)[lagged["row_idx"].to_numpy() + horizon]  # getting only the predictions corresponding to the lagged rows

        res_ar = y - y_ar
        res_pp = y - y_pp

        stat_ar, p_value_ar = shapiro(res_ar)
        stat_pp, p_value_pp = shapiro(res_pp)

        rows.append({
            "dataset": f"Location_{loc}",
            "horizon": horizon,
            "model": "ar_hybrid",
            "stat": float(stat_ar),
            "p_value": float(p_value_ar),
            "reject_normal": p_value_ar < alpha,
        })

        rows.append({
            "dataset": f"Location_{loc}",
            "horizon": horizon,
            "model": "power_prediction",
            "stat": float(stat_pp),
            "p_value": float(p_value_pp),
            "reject_normal": p_value_pp < alpha,
        })

pd.DataFrame(rows)


/home/kuba/Studia/IIst/ProjektNaukowoWdrozeniowy/projekt/ProjektNaukowoWdrozeniowy/.venv/lib/python3.14/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/home/kuba/Studia/IIst/ProjektNaukowoWdrozeniowy/projekt/ProjektNaukowoWdrozeniowy/.venv/lib/python3.14/site-packages/scipy/stats/_axis_nan_policy.py:592: UserWarning: scipy.stats.shapiro: For N > 5000, computed p-value may not be accurate. Current N is 8688.
  res = hypotest_fun_out(*samples, **kwds)
/home/kuba/Studia/IIst/ProjektNaukowoWdrozeniowy/projekt/ProjektNaukowoWdrozeniowy/.venv/lib/python3.14/site-packages/scipy/stats/_axis_nan_policy.py:592: UserWarning: scipy.stats.shapiro: For N > 5000, computed p-value may not be accurate. Current N is 8664.
  res = hypotest_fun_out(*samples, **kwds)
/home/kuba/Studia/IIst/ProjektNaukowoWdrozeniowy/projekt/ProjektNaukowoWdrozeniowy/.venv/lib/python3.14/site-packages/sklearn/u

,dataset,horizon,model,stat,p_value,reject_normal
0,Location_1,24,ar_hybrid,0.987122,2.357561e-27,True
1,Location_1,24,power_prediction,0.987606,7.120706e-27,True
2,Location_1,48,ar_hybrid,0.987228,3.245020e-27,True
3,Location_1,48,power_prediction,0.988784,1.303281e-25,True
4,Location_2,24,ar_hybrid,0.971369,2.202869e-38,True
5,Location_2,24,power_prediction,0.969913,3.920656e-39,True
6,Location_2,48,ar_hybrid,0.973113,2.100493e-37,True
7,Location_2,48,power_prediction,0.970289,6.702408e-39,True
8,Location_3,24,ar_hybrid,0.981162,2.323929e-32,True
9,Location_3,24,power_prediction,0.980599,9.122477e-33,True
